In [237]:
import numpy as np
import c832 as c
import CircuitBuilder
from tqdm import tqdm
import matplotlib.pyplot as plt

In [274]:
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def windowFiller(schedule, nwindow):
    thiswindow = []
    for row in schedule:
        if nwindow in row:
            thiswindow.append(np.where(row == nwindow)[0][0])
        else:
            thiswindow.append(None)
    return thiswindow

def c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule, errorschedule = None):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, 0)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, 0, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, 0, 0, 0)
    cb.al('R 0 1 2 3 4 5 6 7')
    cb.al('H 0 1 2 3 4 5 6 7')
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        #cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Noise rounds
    cb.begin_loop(nrounds)
    timewindow = 0
    for windowz in range(1, np.max(zschedule) + 1):
        cb.time_tick()
        if errorschedule is not None:
            for errornum in range(len(errorschedule[timewindow])):
                if errorschedule[timewindow][errornum] != 'I':
                    cb.inject_error([errornum], errorschedule[timewindow][errornum])
        timewindow += 1
        cb.couple_z_check(windowFiller(zschedule, windowz), True)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        cb.time_tick()
        if errorschedule is not None:
            for errornum in range(len(errorschedule[timewindow])):
                if errorschedule[timewindow][errornum] != 'I':
                    cb.inject_error([errornum], errorschedule[timewindow][errornum])
        timewindow += 1
        cb.couple_x_check(windowFiller(xschedule, windowx), True)
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al('H 0 1 2 3 4 5 6 7')
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb

def c832scheduleableinjectable2(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule, errorschedule = None):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, 0)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, 0, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, 0, 0, 0)
    cb.al('R 0 1 2 3 4 5 6 7')
    cb.al('H 0 1 2 3 4 5 6 7')
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        #cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Noise rounds

    timewindow = 0
    for _ in range(nrounds):
        for windowz in range(1, np.max(zschedule) + 1):
            #cb.time_tick()
            cb.al('TICK')
            if errorschedule is not None:
                for errornum in range(len(errorschedule[timewindow])):
                    if errorschedule[timewindow][errornum] != 'I':
                        cb.inject_error([errornum], errorschedule[timewindow][errornum])
            timewindow += 1
            cb.couple_z_check(windowFiller(zschedule, windowz), True)
        cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
        cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        for windowx in range(1, np.max(xschedule) + 1):
            #cb.time_tick()
            cb.al('TICK')
            if errorschedule is not None:
                for errornum in range(len(errorschedule[timewindow])):
                    if errorschedule[timewindow][errornum] != 'I':
                        cb.inject_error([errornum], errorschedule[timewindow][errornum])
            timewindow += 1
            cb.couple_x_check(windowFiller(xschedule, windowx), True)
        cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
        cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
        cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Cooldown round
    for windowz in range(1, np.max(zschedule) + 1):
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Final measurement of all physical qubits
    cb.al('H 0 1 2 3 4 5 6 7')
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb

def detector_builder(mseq, syndromelength = 5, nphysicalqubits = 8, nrounds = 1, mode = 'binary'):
    if len(mseq) != syndromelength * (nrounds + 1) + nphysicalqubits:
        raise ValueError(f"Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits")
    syndromes, finalmeasurement = mseq[:-nphysicalqubits], mseq[-nphysicalqubits:]
    if mode == 'string':
        det = []
        for i in range(nrounds):
            det.append(''.join([str(0 if x == y else 1) for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])]))
    elif mode == 'binary':
        det = np.zeros(nrounds * syndromelength, dtype = np.bool)
        for i in range(nrounds):
            det[i*syndromelength:(i+1)*syndromelength] = np.array([0 if x == y else 1 for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])])
    else:
        raise ValueError(f"Invalid mode: {mode}")

    return det, finalmeasurement

def check_detector_fired(det):
    for detector in det:
        if '1' in detector:
            return True
    return False

def measure_readout(readout, logical_operator):
    parity = 0
    for i in range(len(readout)):
        if readout[i] == '1' and logical_operator[i]:
            parity += 1
    return parity % 2


In [3]:
zschedule = [[1, 4, 2, 3, 0, 0, 0, 0],
 [0, 0, 0, 0, 2, 3, 1, 4],
 [3, 2, 0, 0, 4, 1, 0, 0],
 [2, 0, 3, 0, 1, 0, 4, 0]]
xschedule = [[1, 0, 4, 0, 0, 3, 0, 2],
 [2, 0, 3, 0, 4, 0, 1, 0],
 [0, 1, 0, 4, 3, 0, 2, 0]]
errorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
]
cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)

In [12]:
nshots = 100000
sample = cb.sample(nshots)
count = 0
for row in tqdm(sample):
    det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0] 
    if '1' not in det[0]:
        count += row[1]
print(count / nshots)

  0%|          | 0/64204 [00:00<?, ?it/s]


ValueError: Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits

In [ ]:
zschedule1 = [[3, 4, 1, 2, 0, 0, 0, 0],
 [0, 0, 0, 0, 4, 3, 2, 1],
 [1, 3, 0, 0, 2, 4, 0, 0],
 [4, 0, 2, 0, 1, 0, 3, 0]]
xschedule1 = [[4, 0, 1, 0, 0, 3, 0, 2],
 [3, 0, 4, 0, 1, 0, 2, 0],
 [0, 1, 0, 2, 4, 0, 3, 0]]
zschedule2 = [[1, 4, 2, 3, 0, 0, 0, 0],
 [0, 0, 0, 0, 2, 3, 1, 4],
 [3, 2, 0, 0, 4, 1, 0, 0],
 [2, 0, 3, 0, 1, 0, 4, 0]]
xschedule2 = [[1, 0, 4, 0, 0, 3, 0, 2],
 [2, 0, 3, 0, 4, 0, 1, 0],
 [0, 1, 0, 4, 3, 0, 2, 0]]

In [20]:

emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
#zschedule, xschedule = zschedule1, xschedule2
errorprob = 0
for errortype in ['X', 'Z', 'Y']:
    for rowcol in tqdm(range(8 * 8)):
        row, col = rowcol // 8, rowcol % 8
        errorschedule = emptyerrorschedule.copy()
        errorschedule[row][col] = errortype
        cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
        nshots = 10000
        sample = cb.sample(nshots)
        count = 0
        for row in sample:
            det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
            if '1' not in det[0]:
                count += row[1]
        errorprob += count / nshots
print(errorprob)

100%|██████████| 64/64 [00:00<00:00, 72.49it/s]

82.0


In [297]:
zschedule, xschedule = zschedule1, xschedule1 # 0.00413
#zschedule, xschedule = zschedule2, xschedule2 # 0.01507
#zschedule, xschedule = zschedule1, xschedule2 # 0.00547
#zschedule, xschedule = zschedule2, xschedule1 # 0.01351
nrounds = 5
cb = c832scheduleableinjectable2(er = 0.001, nrounds = nrounds, zschedule = zschedule, xschedule = xschedule)
sample = cb.sample(10000000)
logop = np.array([1, 1, 0, 0, 1, 1, 0, 0])
s, sandc, sandnc = 0, 0, 0
readoutdict = {}
for row in tqdm(sample):
    det, finalmeasurement = detector_builder(row[0], 7, 8, nrounds + 1, mode = 'string')
    correctreadout = measure_readout(finalmeasurement, logop) == 0
    survives = not check_detector_fired(det)
    if survives:
        s += row[1]
        if finalmeasurement not in readoutdict:
            readoutdict[finalmeasurement] = row[1]
        else:
            readoutdict[finalmeasurement] += row[1]
    if survives and correctreadout:
        sandc += row[1]
print(1 - (sandc / s), s - sandc, s)

100%|██████████| 278056/278056 [00:01<00:00, 167867.38it/s]

5.495278582623442e-06 48 8734771


In [ ]:
readoutdictsorted = sorted(readoutdict.items(), key = lambda x: x[1], reverse = True)
name = 'save2a'
with open(f'savedschedulestats/{name}.txt', 'w') as f:
    f.write(f'{1 - (sandc / s)}, {s - sandc}, {s}\n')
    for i in range(len(zschedule)):
        f.write(f'{zschedule[i]}\n')
    f.write('\n')
    for i in range(len(xschedule)):
        f.write(f'{xschedule[i]}\n')
    f.write('\n')
    f.write(str(cb.get_circuit().diagram()))
    for readout in readoutdictsorted:
        f.write(f'{readout[0]} {readout[1]}\n')
        print(readout[0], readout[1])

10011001 547431
11000011 547318
01100110 546324
01010101 546193
01101001 546063
10100101 546058
11111111 546010
11110000 545899
01011010 545890
10010110 545865
10101010 545479
00110011 545085
00111100 544907
00001111 544875
00000000 544862
11001100 544522
00001010 769
10101111 768
00111001 746
10010011 739
10100000 736
11110101 735
00000101 731
01101100 730
01011111 721
10011100 718
01100011 715
00110110 709
01010000 709
11001001 709
11111010 698
11000110 683


In [287]:
cb.get_circuit().diagram()

/-------------------------------------------------------------------------------------\ /-------\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /-----------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /--------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /-----------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /---------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------\
 q0: -R-H-----@-----------@-------------@----------------------------X---X--------------------------@-----------@-------------@-------------------------------X---X-----------------------------------------------------------------------------@-----------@-------------@-------------------------------X---X--------------------------------------------------------------------------------@-----------@-------------@-------------------------------X-----X---------------------------------------------------------------------------------@-----------@-------------@-------------------------------X---X---------------------------------------------------------------------------------@-----------@-------------@-------------------------------X---X---------------------------------------------------------------------------------@-----------@-------------@-------------------------------X---X------------------------------------------------H----------------------------MR(0):rec[49]-
              |           |             |                            |   |                          |           |             |                               |   |                                                                             |           |             |                               |   |                                                                                |           |             |                               |     |                                                                                 |           |             |                               |   |                                                                                 |           |             |                               |   |                                                                                 |           |             |                               |   |
 q1: -R-H-----|-----------|---@---@-----|------------------X---------|---|--------------------------|-----------|---@---@-----|---------------------X---------|---|-----------------------------------------------------------------------------|-----------|---@---@-----|---------------------X---------|---|--------------------------------------------------------------------------------|-----------|---@---@-----|---------------------X---------|-----|---------------------------------------------------------------------------------|-----------|---@---@-----|---------------------X---------|---|---------------------------------------------------------------------------------|-----------|---@---@-----|---------------------X---------|---|---------------------------------------------------------------------------------|-----------|---@---@-----|---------------------X---------|---|------------------------------------------------H----------------------------MR(0):rec[50]-
              |           |   |   |     |                  |         |   |                          |           |   |   |     |                     |         |   |                                               

In [98]:
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
errorprob = 0
for errortype in ['X', 'Z', 'Y']:
    for rowcol in tqdm(range(15 * 8)):
        row, col = rowcol // 15, rowcol % 8
        errorschedule = emptyerrorschedule.copy()
        errorschedule[row][col] = errortype
        cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
        nshots = 1000
        sample = cb.sample(nshots)
        count = 0
        for row in sample:
            det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
            if '1' not in det[0]:
                count += row[1]
        errorprob += count / nshots
print(errorprob)

100%|██████████| 120/120 [00:00<00:00, 212.06it/s]

196.0


In [97]:
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule2
errorprob = 0
numerrors = 0
errorschedule = emptyerrorschedule.copy()
for rowcol in tqdm(range(15 * 8)):
    row, col = rowcol // 15, rowcol % 15
    randomerror = np.random.choice(['I','X', 'Z', 'Y'], p = [0.99001, 0.00333, 0.00333, 0.00333])
    errorschedule[row][col] = randomerror
    if randomerror != 'I':
        numerrors += 1
        print(row, col, randomerror)
print(np.array(numerrors))
print(np.array(errorschedule))
cb = c832scheduleableinjectable(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
nshots = 1000
sample = cb.sample(nshots)
count = 0
for row in sample:
    det = detector_builder(row[0], 7, 8, 1, mode = 'string')[0]
    if '1' not in det[0]:
        count += row[1]
errorprob += count / nshots
print(errorprob)

100%|██████████| 120/120 [00:00<00:00, 46222.47it/s]

4 4 Y
1
[['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'Y' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']
 ['I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I' 'I']]
0.0


In [83]:
for i in range(8):
    for j in range(15):
        if errorschedule[i][j] != 'I':
            print(i, j, errorschedule[i][j])

6 6 Z
7 5 X


In [76]:
print(cb.get_circuit().diagram())

      /-------------------------------------------------------------------------------------\ /--------------------\ /--------------------\ /--------------------\ /------------------------------------\ /------------------\ /------------------\ /------------------\ /----------------------------------------------------------------------------------------------------\
 q0: -R-H-@-------------@-----@------------------------X-------X------------------------------DEPOLARIZE1(0)-@-------DEPOLARIZE1(0)-------@-DEPOLARIZE1(0)-----@---DEPOLARIZE1(0)-------------------------DEPOLARIZE1(0)-X-----DEPOLARIZE1(0)---X---DEPOLARIZE1(0)-------DEPOLARIZE1(0)------------------------------------------------H--------------------------MR(0):rec[14]-
          |             |     |                        |       |                                             |                            |                    |                                                         |                      |
 q1: -R-H-|----------

In [250]:
nrounds = 5
errorschedulelength = nrounds * 8
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
    for _ in range(errorschedulelength)
]
zschedule, xschedule = zschedule2, xschedule1
numerrors = 0
errorschedule = emptyerrorschedule.copy()
perror = 0.001
thiserrors = []
for rowcol in tqdm(range(errorschedulelength * 15)):
    row, col = rowcol // 15, rowcol % 15
    randomerror = np.random.choice(['I','X', 'Z', 'Y'], p = [1-perror, perror/3, perror/3, perror/3])
    errorschedule[row][col] = randomerror
    if randomerror != 'I':
        numerrors += 1
#        print(row, col, randomerror)
#print(np.array(numerrors))
#print(np.array(errorschedule))
cb = c832scheduleableinjectable2(er = 0, nrounds = nrounds, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
nshots = 4
sample = cb.sample(nshots)
count = 0
errors = []
for row in sample:
    det, finalmeasurement = detector_builder(row[0], 7, 8, nrounds, mode = 'string')
    correctreadout = measure_readout(finalmeasurement, logop) == 0
    survives = not check_detector_fired(det)
    if survives:
        s += row[1]
        if finalmeasurement not in readoutdict:
            readoutdict[finalmeasurement] = row[1]
        else:
            readoutdict[finalmeasurement] += row[1]
    if survives and correctreadout:
        sandc += row[1]
    if survives and not correctreadout:
        errors.append(thiserrors)
#print('no error detected' if count == 0 else f'error detected {count} {nshots}')a

100%|██████████| 600/600 [00:00<00:00, 142364.79it/s]


In [258]:
cb.get_circuit().diagram()

/-------------------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /-----------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /--------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /---------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /---------------------------------------------------------------------------\ /-----\ /-----\ /-----\ /---------------------\ /---\ /---\ /---\ /-----------------------------------------------------------------------------------------\
 q0: -R-H-@-------------@-----@--------------------------------------X---X--------------------@-------------@-----@-----------------------------------------X---X-------------------------------------------------------------------------@-------------@-----@-----------------------------------------X---X----------------------------------------------------------------------------@-------------@-----@-----------------------------------------X---X-----------------------------------------------------------------------------@-------------@-----@-----------------------------------------X---X-----------------------------------------------------------------------------@-------------@-----@-----------------------------------------X---X------------------------------------------------H----------------------------MR(0):rec[42]-
          |             |     |                                      |   |                    |             |     |                                         |   |                                                                         |             |     |                                         |   |                                                                            |             |     |                                         |   |                                                                             |             |     |                                         |   |                                                                             |             |     |                                         |   |
 q1: -R-H-|-----------@-|-----|---@------------------------X---------|---|--------------------|-----------@-|-----|---@---------------------------X---------|---|-------------------------------------------------------------------------|-----------@-|-----|---@---------------------------X---------|---|----------------------------------------------------------------------------|-----------@-|-----|---@---------------------------X---------|---|-----------------------------------------------------------------------------|-----------@-|-----|---@---------------------------X---------|---|-----------------------------------------------------------------------------|-----------@-|-----|---@---------------------------X---------|---|------------------------------------------------H----------------------------MR(0):rec[43]-
          |           | |     |   |                        |         |   |                    |           | |     |   |                           |         |   |                                                                         |           | |     |   |                           |         |   |                                                                            |           | |     |   |                           |         |   |                                                                             |           | |     |   |                           |         |   |                                                                             |           | |     |   |                           |         |   |
 q2: -R-H-|-------@---|-|-----|-@-|--------------------X---|---------

In [288]:
perror, nshots, nrounds = 0.005, 1, 5
errorschedulelength = nrounds * 8
errors = []
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
    for _ in range(errorschedulelength)
]
zschedule, xschedule = zschedule2, xschedule2
simulated = 0
for _ in tqdm(range(50000)):
    numerrors = 0
    errorschedule = emptyerrorschedule.copy()
    thiserrors = []
    for rowcol in range((errorschedulelength - 8) * 15):
        row, col = rowcol // 15, rowcol % 15
        randomerror = np.random.choice(['I','X', 'Z', 'Y'], p = [1-perror, perror/3, perror/3, perror/3])
        errorschedule[row][col] = randomerror
        if randomerror != 'I':
            numerrors += 1
            thiserrors.append((row, col, randomerror))
    if numerrors != 2:
        continue
    simulated += 1
    cb = c832scheduleableinjectable2(er = 0, nrounds = nrounds, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
    sample = cb.sample(nshots)
    count = 0
    for row in sample:
        det, finalmeasurement = detector_builder(row[0], 7, 8, nrounds + 1, mode = 'string')
        correctreadout = measure_readout(finalmeasurement, logop) == 0
        survives = not check_detector_fired(det)
        if survives:
            s += row[1]
            if finalmeasurement not in readoutdict:
                readoutdict[finalmeasurement] = row[1]
            else:
                readoutdict[finalmeasurement] += row[1]
        if survives and correctreadout:
            sandc += row[1]
        if survives and not correctreadout:
            errors.append(thiserrors)
print(simulated, len(errors))

100%|██████████| 50000/50000 [03:25<00:00, 243.50it/s]

13098 56


In [289]:
for error in errors:
    print(error)

[(18, 8, np.str_('Z')), (22, 11, np.str_('Y'))]
[(17, 6, np.str_('Z')), (19, 10, np.str_('Z'))]
[(1, 5, np.str_('Z')), (1, 7, np.str_('Z'))]
[(2, 9, np.str_('Z')), (17, 14, np.str_('Y'))]
[(7, 8, np.str_('Y')), (18, 8, np.str_('Z'))]
[(3, 14, np.str_('Z')), (10, 8, np.str_('Z'))]
[(17, 9, np.str_('Z')), (19, 4, np.str_('Z'))]
[(18, 14, np.str_('X')), (26, 9, np.str_('Z'))]
[(10, 9, np.str_('Z')), (16, 12, np.str_('Z'))]
[(18, 9, np.str_('Z')), (30, 8, np.str_('Y'))]
[(18, 9, np.str_('Z')), (29, 9, np.str_('Z'))]
[(10, 13, np.str_('Z')), (18, 9, np.str_('Z'))]
[(3, 13, np.str_('X')), (18, 8, np.str_('Z'))]
[(25, 1, np.str_('Z')), (28, 3, np.str_('Z'))]
[(1, 12, np.str_('X')), (18, 8, np.str_('Z'))]
[(0, 9, np.str_('Z')), (2, 8, np.str_('Z'))]
[(9, 14, np.str_('X')), (26, 9, np.str_('Z'))]
[(2, 9, np.str_('Z')), (17, 13, np.str_('X'))]
[(12, 0, np.str_('Z')), (12, 2, np.str_('Z'))]
[(3, 12, np.str_('Z')), (10, 9, np.str_('Z'))]
[(10, 8, np.str_('Z')), (20, 11, np.str_('Z'))]
[(2, 9, np.s